In [ ]:
from pathlib import Path
import shutil

ROOT      = Path("..").resolve()
NEG_DIR   = ROOT / "data" / "negatives"
TRAIN_IMG = ROOT / "data" / "processed" / "images" / "train"
TRAIN_LBL = ROOT / "data" / "processed" / "labels" / "train"

# Count what we have
neg_count = len(list(NEG_DIR.rglob("*.jpg"))) + len(list(NEG_DIR.rglob("*.png")))
print(f"Train images    : {len(list(TRAIN_IMG.glob('*.jpg'))):,}")
print(f"Train labels    : {len(list(TRAIN_LBL.glob('*.txt'))):,}")
print(f"Negatives ready : {neg_count}")
print()
# Breakdown by category
for folder in sorted(NEG_DIR.iterdir()):
    if folder.is_dir():
        count = len(list(folder.glob("*.*")))
        print(f"  {folder.name:12}: {count} images")

In [ ]:
added    = 0
skipped  = 0

for img_path in NEG_DIR.rglob("*"):
    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    dst_img = TRAIN_IMG / f"neg_{img_path.stem}.jpg"
    dst_lbl = TRAIN_LBL / f"neg_{img_path.stem}.txt"

    if dst_img.exists():
        skipped += 1
        continue

    shutil.copy2(str(img_path), str(dst_img))
    dst_lbl.write_text("")  # empty label = no faces
    added += 1

print(f"Added   : {added}")
print(f"Skipped : {skipped} (already existed)")

In [ ]:
total_imgs   = len(list(TRAIN_IMG.glob("*.jpg")))
empty_labels = sum(1 for f in TRAIN_LBL.glob("*.txt") 
                   if f.read_text().strip() == "")
face_labels  = total_imgs - empty_labels
ratio        = empty_labels / total_imgs

print(f"Total images    : {total_imgs:,}")
print(f"Face images     : {face_labels:,}  ({face_labels/total_imgs*100:.1f}%)")
print(f"Negative images : {empty_labels:,}  ({empty_labels/total_imgs*100:.1f}%)")
print(f"Negative ratio  : {ratio:.3f}")
print()
if ratio < 0.05:
    print("⚠️  Need more negatives — target 10-20%")
elif ratio > 0.30:
    print("⚠️  Too many negatives — might hurt recall")
else:
    print("✅ Good balance for V2 training")

In [ ]:
import urllib.request
import json

# Download 1500 negative images from COCO val2017
# These are already curated, clean images - animals, objects, scenes

COCO_ANNO_URL = "https://huggingface.co/datasets/merve/coco/resolve/main/annotations/instances_val2017.json"
anno_path = ROOT / "data" / "negatives" / "coco_instances_val2017.json"

if not anno_path.exists():
    print("Downloading COCO annotations (small JSON)...")
    urllib.request.urlretrieve(COCO_ANNO_URL, str(anno_path))
    print("Done")
else:
    print("Already exists")

with open(anno_path) as f:
    coco = json.load(f)

# Find person category ID
person_id = next(c["id"] for c in coco["categories"] if c["name"] == "person")

# Get image IDs that have NO person annotations
person_img_ids = {a["image_id"] for a in coco["annotations"] 
                  if a["category_id"] == person_id}
no_person_imgs = [img for img in coco["images"] 
                  if img["id"] not in person_img_ids]

print(f"Person category ID       : {person_id}")
print(f"Images with person       : {len(person_img_ids):,}")
print(f"Images WITHOUT person    : {len(no_person_imgs):,}")
print(f"We will download 1,100 of these")


In [ ]:
import random
import time

random.seed(42)
selected = random.sample(no_person_imgs, 1100)

auto_neg_dir = ROOT / "data" / "negatives" / "auto"
auto_neg_dir.mkdir(exist_ok=True)

already = len(list(auto_neg_dir.glob("*.jpg")))
print(f"Already downloaded: {already}")

downloaded = already
failed     = 0

for i, img_info in enumerate(selected):
    dst = auto_neg_dir / img_info["file_name"]
    if dst.exists():
        continue
    try:
        urllib.request.urlretrieve(img_info["coco_url"], str(dst))
        downloaded += 1
        if downloaded % 100 == 0:
            print(f"  {downloaded}/1100 downloaded...")
    except:
        failed += 1

print(f"\n✅ Done — Downloaded: {downloaded}, Failed: {failed}")

In [ ]:
added   = 0
skipped = 0

for img_path in auto_neg_dir.glob("*.jpg"):
    dst_img = TRAIN_IMG / f"autoneg_{img_path.name}"
    dst_lbl = TRAIN_LBL / f"autoneg_{img_path.stem}.txt"

    if dst_img.exists():
        skipped += 1
        continue

    shutil.copy2(str(img_path), str(dst_img))
    dst_lbl.write_text("")
    added += 1

print(f"Added   : {added}")
print(f"Skipped : {skipped}")

# Final balance check
total      = len(list(TRAIN_IMG.glob("*.jpg")))
empty      = sum(1 for f in TRAIN_LBL.glob("*.txt") 
                 if f.read_text().strip() == "")
ratio      = empty / total

print(f"\nFinal dataset:")
print(f"  Total images    : {total:,}")
print(f"  Face images     : {total-empty:,}  ({(total-empty)/total*100:.1f}%)")
print(f"  Negative images : {empty:,}  ({ratio*100:.1f}%)")

if 0.08 <= ratio <= 0.25:
    print("\n✅ Good balance — ready for V2 training")
else:
    print(f"\n⚠️  Ratio {ratio:.2f} — adjust if needed")

In [ ]:
import subprocess

zip_path = ROOT / "dataset_v2.zip"

print("Zipping dataset_v2... (this takes 2-3 mins)")
result = subprocess.run(
    ["python", "-m", "zipfile", "-c", 
     str(zip_path),
     str(ROOT / "data" / "processed"),
     str(ROOT / "data" / "face_dataset.yaml")],
    capture_output=True, text=True
)

size_mb = zip_path.stat().st_size / 1024 / 1024
print(f"✅ Done — dataset_v2.zip = {size_mb:.0f} MB")
print(f"Location: {zip_path}")
